# 📝 Week 4 Assignment: Data Analysis & Visualization
# Анализ и визуализация данных

---

These exercises cover:
- NumPy for numerical operations
- Pandas for tabular data
- Matplotlib/Seaborn for visualization
- Real bioinformatics data analysis

---

<!-- COPILOT_NOTEBOOK_ONBOARDING_V1 -->
## Why this notebook matters in real projects
This assignment notebook is designed for hands-on practice of production-relevant analysis patterns.


## How to start using this notebook
1. Run the **Quick starter data demo** cell below first to confirm your environment and file paths.
2. Then run notebook sections top-to-bottom once, and only then start modifying parameters.
3. Keep one small, known-good example input while experimenting; compare each output against it.
4. For larger or real data, copy the same workflow and replace only input paths first.


## Complicated moments explained
These are common points where learners usually get stuck:
- Start with tiny inputs first, then scale after outputs look correct.
- Document assumptions for every threshold or filtering decision.
- Debug shape/type/path issues early to avoid cascading errors.


## Quick starter data demo (run this first)
This cell loads tiny synthetic files from `Course/Assets/data/notebook_starters/` so you can see real input/output behavior immediately.


In [ ]:
# Quick starter demo: load tiny example files and inspect outputs
from pathlib import Path
import json
import csv

root = Path.cwd().resolve()
while root != root.parent and not (root / 'Course').exists():
    root = root.parent

base = root / 'Course' / 'Assets' / 'data' / 'notebook_starters'
print(f'Using starter data folder: {base}')

with open(base / 'dna_sequences.csv', newline='') as f:
    rows = list(csv.DictReader(f))
print('dna_sequences.csv (first 3 rows):')
for row in rows[:3]:
    print(row)

with open(base / 'variants.csv', newline='') as f:
    variants = list(csv.DictReader(f))
print('\nvariants.csv (first 3 rows):')
for row in variants[:3]:
    print(row)

print('\nprotein_sequences.fasta (headers):')
for line in (base / 'protein_sequences.fasta').read_text().splitlines():
    if line.startswith('>'):
        print(line)

meta = json.loads((base / 'sample_metadata.json').read_text())
print('\nMetadata keys:', list(meta.keys()))


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Style settings
plt.style.use('seaborn-whitegrid')
%matplotlib inline

## 📊 Part 1: NumPy for Sequence Analysis

### Problem 1.1: Sequence Encoding

Encode DNA sequences as numerical arrays for analysis.

In [ ]:
def encode_sequence(sequence):
    """
    Convert DNA sequence to one-hot encoded numpy array.
    
    Shape: (sequence_length, 4) where columns are [A, T, G, C]
    
    Example:
    'ATG' -> [[1,0,0,0], [0,1,0,0], [0,0,1,0]]
    """
    nucleotides = {'A': 0, 'T': 1, 'G': 2, 'C': 3}
    # YOUR CODE HERE
    pass

def decode_sequence(encoded):
    """Convert one-hot encoded array back to string"""
    nucleotides = ['A', 'T', 'G', 'C']
    # YOUR CODE HERE
    pass

# Test
seq = "ATGCATGC"
encoded = encode_sequence(seq)
print(f"Shape: {encoded.shape}")
print(f"Decoded: {decode_sequence(encoded)}")

### Problem 1.2: Position Weight Matrix (PWM)

Create a PWM from aligned sequences.

In [ ]:
def create_pwm(sequences):
    """
    Create Position Weight Matrix from aligned sequences.
    
    Args:
        sequences: List of equal-length DNA sequences
    
    Returns:
        PWM as numpy array (4 x sequence_length)
        Rows: A, T, G, C
    """
    # YOUR CODE HERE
    pass

def score_sequence(sequence, pwm):
    """
    Score a sequence against a PWM.
    
    Returns:
        Log-odds score
    """
    # YOUR CODE HERE
    pass

# Test with TATA box sequences
tata_sequences = [
    "TATAAAA",
    "TATAAAT",
    "TATAAAG",
    "TATATAA",
    "TATAAAA"
]

pwm = create_pwm(tata_sequences)
print("PWM:")
print(pwm)

# Score test sequences
test_seqs = ["TATAAAA", "GCGCGCG", "TATATTA"]
for seq in test_seqs:
    print(f"{seq}: {score_sequence(seq, pwm):.2f}")

### Problem 1.3: GC Content Sliding Window

Use NumPy for efficient GC content calculation.

In [ ]:
def gc_sliding_window_numpy(sequence, window_size):
    """
    Calculate GC content in sliding windows using NumPy.
    
    Returns:
        Array of GC percentages for each window position
    """
    # Convert sequence to binary array (1 for G/C, 0 for A/T)
    # YOUR CODE HERE
    pass

# Test
seq = "ATGCGCGCATATATATGCGCGC" * 10
gc_values = gc_sliding_window_numpy(seq, 20)
print(f"GC values shape: {gc_values.shape}")
print(f"Mean GC: {gc_values.mean():.1f}%")

---

## 📊 Part 2: Pandas for Gene Expression

### Problem 2.1: Load and Explore Expression Data

Create and analyze a gene expression DataFrame.

In [ ]:
# Create sample expression data
np.random.seed(42)
n_genes = 100
n_samples = 6

# Generate gene names and sample names
genes = [f"Gene_{i:04d}" for i in range(n_genes)]
samples = ['Control_1', 'Control_2', 'Control_3', 
           'Treatment_1', 'Treatment_2', 'Treatment_3']

# Generate expression values (log2 scale)
# Some genes will be differentially expressed
base_expression = np.random.normal(8, 2, (n_genes, n_samples))

# Make some genes differentially expressed
de_genes = np.random.choice(n_genes, 20, replace=False)
base_expression[de_genes, 3:] += np.random.choice([-2, 2], (20, 3))

expression_df = pd.DataFrame(base_expression, 
                             index=genes, 
                             columns=samples)

print(expression_df.head())

In [ ]:
# Problem 2.1a: Calculate basic statistics

def expression_summary(df):
    """
    Calculate summary statistics for expression data.
    
    Returns:
        DataFrame with: mean, std, min, max for each gene
    """
    # YOUR CODE HERE
    pass

summary = expression_summary(expression_df)
print(summary.head())

### Problem 2.2: Differential Expression Analysis

In [ ]:
from scipy import stats

def find_de_genes(df, control_cols, treatment_cols, 
                  fc_threshold=1.5, pvalue_threshold=0.05):
    """
    Find differentially expressed genes.
    
    Args:
        df: Expression DataFrame
        control_cols: List of control sample columns
        treatment_cols: List of treatment sample columns
        fc_threshold: Minimum fold change
        pvalue_threshold: Maximum p-value
    
    Returns:
        DataFrame with: gene, log2FC, pvalue, significant
    """
    results = []
    
    for gene in df.index:
        control = df.loc[gene, control_cols].values
        treatment = df.loc[gene, treatment_cols].values
        
        # Calculate log2 fold change
        log2fc = treatment.mean() - control.mean()  # Already log2
        
        # T-test
        _, pvalue = stats.ttest_ind(control, treatment)
        
        results.append({
            'gene': gene,
            'log2FC': log2fc,
            'pvalue': pvalue,
            'significant': (abs(log2fc) >= fc_threshold) & (pvalue < pvalue_threshold)
        })
    
    return pd.DataFrame(results)

# Find DE genes
de_results = find_de_genes(
    expression_df,
    ['Control_1', 'Control_2', 'Control_3'],
    ['Treatment_1', 'Treatment_2', 'Treatment_3']
)

print(f"Total significant genes: {de_results['significant'].sum()}")
print(de_results[de_results['significant']].head())

### Problem 2.3: Gene Filtering and Grouping

In [ ]:
def filter_by_expression(df, min_mean=5, max_cv=0.5):
    """
    Filter genes by expression criteria.
    
    Args:
        df: Expression DataFrame
        min_mean: Minimum mean expression
        max_cv: Maximum coefficient of variation
    
    Returns:
        Filtered DataFrame
    """
    # YOUR CODE HERE
    pass

def cluster_genes(df, n_clusters=3):
    """
    Cluster genes by expression pattern.
    
    Returns:
        DataFrame with cluster assignments
    """
    from sklearn.cluster import KMeans
    # YOUR CODE HERE
    pass

# Test
filtered = filter_by_expression(expression_df)
print(f"Genes after filtering: {len(filtered)}")

---

## 📊 Part 3: Visualization

### Problem 3.1: GC Content Landscape Plot

In [ ]:
def plot_gc_landscape(sequence, window_size=100, step=10):
    """
    Create a GC content landscape plot.
    
    Similar to the GC_landscape project from source materials.
    """
    # Calculate GC in windows
    positions = []
    gc_values = []
    
    for i in range(0, len(sequence) - window_size, step):
        window = sequence[i:i+window_size]
        gc = (window.count('G') + window.count('C')) / window_size * 100
        positions.append(i + window_size // 2)
        gc_values.append(gc)
    
    # Create plot
    fig, ax = plt.subplots(figsize=(12, 4))
    
    # YOUR CODE HERE: Create a filled line plot
    # Add mean line and standard deviation bands
    
    ax.set_xlabel('Position (bp)')
    ax.set_ylabel('GC Content (%)')
    ax.set_title(f'GC Content Landscape (window={window_size}bp)')
    
    plt.tight_layout()
    return fig, ax

# Generate test sequence with varying GC
test_seq = "ATATATAT" * 50 + "GCGCGCGC" * 50 + "ATATATAT" * 50
plot_gc_landscape(test_seq, window_size=50, step=5)
plt.show()

### Problem 3.2: Volcano Plot

In [ ]:
def volcano_plot(de_results, fc_threshold=1.5, pvalue_threshold=0.05):
    """
    Create a volcano plot for differential expression.
    
    Args:
        de_results: DataFrame with log2FC and pvalue columns
        fc_threshold: Fold change cutoff for significance
        pvalue_threshold: P-value cutoff for significance
    """
    fig, ax = plt.subplots(figsize=(10, 8))
    
    # Calculate -log10(pvalue)
    de_results = de_results.copy()
    de_results['neg_log_pvalue'] = -np.log10(de_results['pvalue'] + 1e-10)
    
    # YOUR CODE HERE:
    # 1. Plot non-significant genes in gray
    # 2. Plot upregulated genes in red
    # 3. Plot downregulated genes in blue
    # 4. Add threshold lines
    # 5. Label significant genes
    
    ax.set_xlabel('log2 Fold Change')
    ax.set_ylabel('-log10(p-value)')
    ax.set_title('Differential Expression Volcano Plot')
    
    plt.tight_layout()
    return fig, ax

# Test
volcano_plot(de_results)
plt.show()

### Problem 3.3: Expression Heatmap

In [ ]:
def expression_heatmap(df, n_genes=50, cluster=True):
    """
    Create a heatmap of gene expression.
    
    Args:
        df: Expression DataFrame
        n_genes: Number of most variable genes to show
        cluster: Whether to cluster rows and columns
    """
    # Select most variable genes
    gene_var = df.var(axis=1).sort_values(ascending=False)
    top_genes = gene_var.head(n_genes).index
    subset = df.loc[top_genes]
    
    # Z-score normalize
    z_scored = (subset.T - subset.mean(axis=1)) / subset.std(axis=1)
    z_scored = z_scored.T
    
    # YOUR CODE HERE: Create clustered heatmap using seaborn
    # Use sns.clustermap for hierarchical clustering
    
    pass

# Test
expression_heatmap(expression_df, n_genes=30)

### Problem 3.4: Protein Length Distribution Histogram

Recreate the protein length histogram from the ORF finder project.

In [ ]:
def plot_protein_lengths(protein_lengths, bins='auto', min_length=0):
    """
    Plot histogram of protein lengths.
    
    Similar to the ORF_finder visualization.
    
    Args:
        protein_lengths: List of protein lengths in amino acids
        bins: Number of bins or 'auto'
        min_length: Minimum length to include
    """
    # Filter by minimum length
    lengths = [l for l in protein_lengths if l >= min_length]
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # YOUR CODE HERE:
    # ax1: Regular histogram
    # ax2: Log-scale histogram
    
    plt.tight_layout()
    return fig, (ax1, ax2)

# Generate sample data
np.random.seed(42)
protein_lengths = list(np.random.exponential(150, 1000).astype(int))
protein_lengths = [l for l in protein_lengths if l > 0]

plot_protein_lengths(protein_lengths, min_length=30)
plt.show()

### Problem 3.5: Codon Usage Bar Chart

In [ ]:
def plot_codon_usage(codon_counts, amino_acid=None):
    """
    Plot codon usage frequencies.
    
    Args:
        codon_counts: Dictionary of codon counts
        amino_acid: If provided, show only codons for this AA
    """
    # YOUR CODE HERE:
    # Create a bar chart showing codon usage
    # Group by amino acid with different colors
    pass

# Sample codon usage data
codon_usage = {
    'UUU': 215, 'UUC': 192, 'UUA': 120, 'UUG': 132,
    'CUU': 105, 'CUC': 109, 'CUA': 34, 'CUG': 550,
    'AUU': 290, 'AUC': 261, 'AUA': 51, 'AUG': 299
}

plot_codon_usage(codon_usage)

---

## 📊 Part 4: Complete Analysis Project

### Problem 4: Gene Expression Analysis Pipeline

Combine everything into a complete analysis.

In [ ]:
class ExpressionAnalysis:
    """
    Complete gene expression analysis pipeline.
    
    Methods:
    - load_data()
    - quality_control()
    - normalize()
    - differential_expression()
    - visualize_results()
    - export_results()
    """
    
    def __init__(self, expression_df, sample_info=None):
        self.raw_data = expression_df
        self.normalized_data = None
        self.de_results = None
        self.sample_info = sample_info
    
    def quality_control(self):
        """
        Perform QC checks on expression data.
        
        Returns:
            Dictionary with QC metrics
        """
        # YOUR CODE HERE:
        # - Check for missing values
        # - Check expression distribution
        # - Check sample correlations
        pass
    
    def normalize(self, method='zscore'):
        """
        Normalize expression data.
        
        Args:
            method: 'zscore', 'quantile', or 'log2'
        """
        # YOUR CODE HERE
        pass
    
    def differential_expression(self, control_samples, treatment_samples):
        """
        Perform differential expression analysis.
        """
        # YOUR CODE HERE
        pass
    
    def visualize_results(self):
        """
        Create comprehensive visualization of results.
        
        Returns:
            Matplotlib figure with multiple panels
        """
        fig, axes = plt.subplots(2, 2, figsize=(14, 12))
        
        # YOUR CODE HERE:
        # Panel 1: Sample correlation heatmap
        # Panel 2: Volcano plot
        # Panel 3: Top DE genes heatmap
        # Panel 4: MA plot
        
        plt.tight_layout()
        return fig
    
    def export_results(self, filename):
        """
        Export results to CSV.
        """
        # YOUR CODE HERE
        pass

# Test the pipeline
analysis = ExpressionAnalysis(expression_df)
qc = analysis.quality_control()
print(qc)

---

## 📚 Summary

In this assignment you practiced:

1. **NumPy**: Sequence encoding, PWMs, efficient calculations
2. **Pandas**: Expression data, filtering, grouping, statistics
3. **Visualization**: GC landscapes, volcano plots, heatmaps, histograms
4. **Integration**: Complete analysis pipelines

These tools are essential for any bioinformatics data analysis!

---